# CTCL Visium — CondSCVI → DestVI deconvolution

Per-spot cell-type deconvolution of the ingested CTCL Visium object from
`00_preprocess/03_spatial_visium_download.ipynb`, using the Li-2024 scRNA skin atlas as reference.

**Pipeline:** `CondSCVI` (cell-type-conditioned scVI on the scRNA reference) → `DestVI`
(`from_rna_model`) deconvolves each Visium spot into cell-type **proportions**. nb-23 cross-patient
subclonal **programs** + an overall malignant signature are then scored on tumour-cell-specific
expression (`get_scale_for_ct("tumor_cell")`), so the malignant signal is restricted to the tumour
compartment rather than the bulk spot.

**Cohort:** 23 sections in two conditions — **8 CTCL lesional** (`CTCL1..8`) and **15 healthy
control** skin (`WS_D_SKN*`). Descriptive results contrast the two; spatial panels annotate every
CTCL section (+ one healthy negative control).

**Structure:** config → data + shared gene space → **train/load models (cached)** → inference →
descriptive (CTCL vs healthy) → spatial annotation. Models/derived h5ad gated by `exists()`; flip
`FORCE_TRAIN` / `FORCE_INFER` in §0 to regenerate.

> ⚠️ **Resolution caveat:** DestVI assumes ~1 cell per observation; 55 µm Visium spots are
> multi-cell, so proportions are spot-level mixtures, not single-cell calls.

> Heavy (load / train / `get_*`) → run on the **GPU kernel**, not the login node.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

import scvi
from scvi.model import CondSCVI, DestVI

SEED = 0
scvi.settings.seed = SEED
sc.settings.set_figure_params(dpi=80, facecolor="white", figsize=(5, 3))

# --- cache toggles ---------------------------------------------------------------
# ONE-TIME regeneration after the spot-QC fix: DestVI + derived h5ad rebuild on the clean cohort.
# After this run finishes, set FORCE_DESTVI and FORCE_INFER back to False (CondSCVI cache is reused).
FORCE_TRAIN  = False   # True -> retrain CondSCVI AND DestVI even if a saved model exists
FORCE_DESTVI = True    # True -> retrain DestVI only (after changing spot QC); CondSCVI cache reused
FORCE_INFER  = True    # True -> recompute the derived h5ad (proportions / program scores)


def _resolve_nb_dir():
    start = Path(__file__).parent.resolve() if "__file__" in globals() else Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(f"could not locate MF/data starting from {start}")


NB_DIR     = _resolve_nb_dir()
FIG_DIR    = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
sc.settings.figdir = str(FIG_DIR)
VISIUM_DIR = NB_DIR / "data" / "Li2024_atlas" / "visium"

# inputs
VISIUM_H5AD = VISIUM_DIR / "ctcl_visium.h5ad"                                  # from nb 24
REF_H5AD    = NB_DIR / "data" / "CTCL_all_final_portal_tags.h5ad"              # scRNA ref (broad cell_type)
SIGN_JSON   = NB_DIR / "data" / "atlas_joint" / "subclone_signatures_v2.json"  # from nb 23
# cached models
CONDSCVI_DIR = VISIUM_DIR / "condscvi_model"
DESTVI_DIR   = VISIUM_DIR / "destvi_model"
# derived output
DECONV_H5AD  = VISIUM_DIR / "ctcl_visium_destvi.h5ad"


def load_or_train(cache_dir, load_fn, train_fn, *, force=False, label=""):
    """Load a saved scvi model if present, else train + save (repo caching idiom).

    load_fn() -> model (cache dir already exists); train_fn() -> a freshly trained model.
    Falls back to train_fn on any load failure. Returns (model, trained) so callers can
    guard convergence plots (a pure load has empty .history).
    """
    cache_dir = Path(cache_dir)
    if cache_dir.exists() and not force:
        try:
            m = load_fn()
            print(f"loaded cached {label} <- {cache_dir}")
            return m, False
        except Exception as exc:  # noqa: BLE001
            print(f"load {label} failed ({exc}); retraining")
    m = train_fn()
    cache_dir.parent.mkdir(parents=True, exist_ok=True)
    m.save(str(cache_dir), overwrite=True)
    print(f"trained + saved {label} -> {cache_dir}")
    return m, True


print("scvi", scvi.__version__, "| NB_DIR", NB_DIR)

## 1. Load Visium + spot QC

Read `ctcl_visium.h5ad` once (raw counts in `layers['counts']`; `condition` labels each spot CTCL or
healthy). **Spot QC:** the healthy `WS_D_SKN*` sections were ingested as the full 4,992-spot capture
grid with empty background spots **not** removed (median ~193 UMIs), whereas the CTCL sections are
already tissue-filtered (median ~8.6k UMIs). Left unfiltered, ~85% of the object is near-empty
background that dominates and corrupts the DestVI fit. So drop spots below `MIN_COUNTS` UMIs
**consistently across both cohorts** before deconvolution. DestVI then operates per-spot on the full
shared gene space — no HVG subset or batch key needed.

In [ ]:
vis_full = sc.read_h5ad(VISIUM_H5AD)        # full gene space; shared subset built in §2
print(vis_full)

MIN_COUNTS = 1000                            # drop near-empty background spots (both cohorts, consistent)
n0 = vis_full.n_obs
vis_full = vis_full[vis_full.obs["total_counts"] >= MIN_COUNTS].copy()
print(f"\nspot QC total_counts>={MIN_COUNTS}: {n0} -> {vis_full.n_obs} ({n0 - vis_full.n_obs} dropped)")
print("condition after QC:", vis_full.obs["condition"].value_counts().to_dict())
_sec = (vis_full.obs[["section", "condition"]]
        .value_counts().rename("n_spots").reset_index()
        .sort_values(["condition", "n_spots"], ascending=[True, False]))
print("\nspots per section:\n", _sec.to_string(index=False))

## 2. Train or load models (cached)

Both models are gated by `exists() and not FORCE_TRAIN` via `load_or_train` — a rerun loads the saved
`.pt` instead of retraining. Convergence curves are drawn only when a model actually trained this run.

### 2a. CondSCVI — cell-type-conditioned scVI on the scRNA reference

Loads the Li-2024 skin atlas, collapses its 49 fine labels → ~14 readable groups (tumour + T-cell
axes kept separate), restricts to the gene space **shared with the Visium** (DestVI needs the same
genes), balanced-subsamples per group (cap 5000) for tractability, then trains `CondSCVI`. This is
the reference DestVI (§2b) deconvolves against.

> Loads a ~9.6 GB atlas → **GPU kernel only.**

In [ ]:
# collapse the 49 fine atlas labels -> ~14 readable groups (keep tumour + T-cell axes separate).
CT_GROUP = {
    "tumor_cell": "tumor_cell", "Th": "Th", "Treg": "Treg",
    "Tc": "CD8_T", "Tc17_Th17": "CD8_T", "Tc_IL13_IL22": "CD8_T",
    "NK": "NK_ILC", "ILC1_3": "NK_ILC", "ILC1_NK": "NK_ILC", "ILC2": "NK_ILC",
    "B_cell": "B_Plasma", "Plasma": "B_Plasma",
    "DC1": "Myeloid", "DC2": "Myeloid", "MigDC": "Myeloid", "pDC": "Myeloid",
    "moDC_1": "Myeloid", "moDC_2": "Myeloid", "moDC_3": "Myeloid",
    "LC_1": "Myeloid", "LC_2": "Myeloid", "LC_3": "Myeloid", "LC_4": "Myeloid",
    "Macro_1": "Myeloid", "Macro_2": "Myeloid", "Mono_mac": "Myeloid", "Inf_mac": "Myeloid",
    "Mast_cell": "Mast",
    "Basal": "Keratinocyte", "basal2": "Keratinocyte", "Differentiated_KC": "Keratinocyte",
    "Differentiated_KC*": "Keratinocyte", "Undifferentiated_KC": "Keratinocyte",
    "Proliferating_KC": "Keratinocyte", "Sebaceous": "Keratinocyte",
    "F1": "Fibroblast", "F2": "Fibroblast", "F3": "Fibroblast",
    "VE1": "Endothelial", "VE2": "Endothelial", "VE3": "Endothelial",
    "LE1": "Endothelial", "LE2": "Endothelial",
    "Pericyte_1": "Pericyte", "Pericyte_2": "Pericyte",
    "Melanocyte": "Melanocyte", "Schwann_1": "Schwann",
    # dropped (ambiguous / non-cell): "channel", "immune"
}

ref = sc.read_h5ad(REF_H5AD)
ref.obs["ct_group"] = ref.obs["cell_type"].astype(str).map(CT_GROUP)
n_drop = int(ref.obs["ct_group"].isna().sum())
ref = ref[ref.obs["ct_group"].notna()].copy()
ref.obs["ct_group"] = ref.obs["ct_group"].astype("category")
print(f"reference: {ref.shape} | dropped {n_drop} cells (channel/immune/unmapped)")

# balanced subsample per group (cap for CondSCVI tractability).
CAP = 5000
rng = np.random.default_rng(SEED)
grp = ref.obs["ct_group"].values
keep_idx = [rng.choice(np.where(grp == g)[0], min(CAP, int((grp == g).sum())), replace=False)
            for g in ref.obs["ct_group"].cat.categories]
ref = ref[np.sort(np.concatenate(keep_idx))].copy()

# shared FULL gene space (ref order) for both CondSCVI and DestVI -- must precede CondSCVI training.
genes = [g for g in ref.var_names if g in set(vis_full.var_names)]   # ~15790 (all ref genes ⊂ visium)
ref = ref[:, genes].copy()
vis = vis_full[:, genes].copy()                                     # spot QC already applied in §1
print(f"subsampled ref: {ref.shape} | shared genes: {len(genes)} | visium: {vis.shape}")
print("groups:", ref.obs["ct_group"].value_counts().to_dict())

# Stage 1 -- CondSCVI on the scRNA reference.
CondSCVI.setup_anndata(ref, labels_key="ct_group", layer="raw_counts")


def _train_condscvi():
    m = CondSCVI(ref)
    m.train(max_epochs=200, accelerator="gpu")
    return m


sc_model, condscvi_trained = load_or_train(
    CONDSCVI_DIR,
    load_fn=lambda: CondSCVI.load(str(CONDSCVI_DIR), adata=ref),
    train_fn=_train_condscvi, force=FORCE_TRAIN, label="CondSCVI",
)

if condscvi_trained:
    ax = sc_model.history["elbo_train"].plot(logy=True, legend=False)
    ax.set(title="CondSCVI ELBO (should flatten)"); plt.tight_layout(); plt.show()

### 2b. DestVI — deconvolve the Visium spots against the CondSCVI reference

`from_rna_model` bakes the CondSCVI decoder + cell-type mapping into DestVI, so on a **cache hit
`DestVI.load` needs only the Visium adata** (not the CondSCVI model). Trains long (2000 epochs) — the
cache makes that a one-time cost.

In [ ]:
# Stage 2 -- DestVI: deconvolve the Visium spots (shared gene space; built in §2a).
DestVI.setup_anndata(vis, layer="counts")


def _train_destvi():
    m = DestVI.from_rna_model(vis, sc_model)
    m.train(max_epochs=2000, n_epochs_kl_warmup=200, accelerator="gpu")
    return m


destvi, destvi_trained = load_or_train(
    DESTVI_DIR,
    load_fn=lambda: DestVI.load(str(DESTVI_DIR), adata=vis),
    train_fn=_train_destvi, force=FORCE_TRAIN or FORCE_DESTVI, label="DestVI",
)

## 3. DestVI inference — cell-type proportions + subclonal-program scoring

Cached to `ctcl_visium_destvi.h5ad` (`exists() and not FORCE_INFER`). Per-spot proportions
(`get_proportions`), then nb-23 **cross-patient** subclonal programs + the overall malignant
signature scored on **tumour-cell-specific** expression (`get_scale_for_ct("tumor_cell")`), so the
program signal is restricted to the malignant compartment rather than the bulk spot. Run nb 23 first
so `SIGN_JSON` exists.

In [ ]:
if DECONV_H5AD.exists() and not FORCE_INFER:
    vis = sc.read_h5ad(DECONV_H5AD)
    print("loaded cached DestVI inference <-", DECONV_H5AD)
else:
    props = destvi.get_proportions()                     # n_spots x n_groups (rows sum to 1)
    vis.obsm["ct_proportions"] = props.values
    for g in props.columns:
        vis.obs[f"ct_{g}"] = props[g].values
    vis.obs["ct_argmax"] = props.idxmax(axis=1).astype("category")

    # nb-23 subclonal signatures on TUMOUR-cell-specific expression (px_scale), not the bulk spot.
    sigs = json.loads(SIGN_JSON.read_text())
    tumor_expr = destvi.get_scale_for_ct("tumor_cell")   # n_spots x n_genes, index=vis.obs_names
    tad = sc.AnnData(tumor_expr.values.astype("float32"),
                     obs=vis.obs[["section"]].copy(),
                     var=pd.DataFrame(index=tumor_expr.columns))
    sc.pp.log1p(tad)
    scored, prog_cols = [], []
    for name, gset in sigs.items():
        present = [g for g in gset if g in tad.var_names]
        if len(present) < 5:
            continue
        sc.tl.score_genes(tad, present, score_name=name, random_state=SEED)
        vis.obs[name] = tad.obs[name].values
        scored.append(name)
        if name.startswith("program__"):
            prog_cols.append(name)
    vis.obs["malignant_score"] = vis.obs["malignant_overall"] if "malignant_overall" in vis.obs else np.nan
    if prog_cols:   # dominant subclonal PROGRAM per spot (cross-patient divergence axis)
        vis.obs["program_argmax"] = (vis.obs[prog_cols].idxmax(axis=1)
                                     .str.replace("program__", "", regex=False).astype("category"))
    print(f"scored {len(scored)} signatures | programs: {[c.replace('program__', '') for c in prog_cols]}")
    vis.write_h5ad(DECONV_H5AD)
    print("saved DestVI inference ->", DECONV_H5AD)

print("proportions:", vis.obsm["ct_proportions"].shape,
      "| dominant-type spread:", vis.obs["ct_argmax"].value_counts().to_dict())

## 4. Descriptive results — CTCL vs healthy

Positive-control the deconvolution: tumour-cell proportion and the malignant signature should
concentrate in the 8 CTCL sections, not the 15 healthy controls. Then the cell-type composition
shift and which subclonal programs are active in the CTCL tumour compartment. One figure per
question; reads the cached §3 adata.

In [ ]:
obs  = vis.obs
cond = obs["condition"].astype(str)
ct_cols   = [c for c in obs.columns if c.startswith("ct_") and c != "ct_argmax"]
groups    = [c[3:] for c in ct_cols]
prog_cols = [c for c in obs.columns if c.startswith("program__")]
ORDER = ["CTCL", "healthy"]

# 1. mean cell-type composition by condition
comp = obs.groupby(cond, observed=True)[ct_cols].mean().T
comp.index = groups
ax = comp[ORDER].plot.bar(figsize=(6, 3.2))
ax.set(ylabel="mean spot proportion", xlabel="",
       title="CTCL spots enriched for tumour + T/myeloid vs healthy")
plt.xticks(rotation=45, ha="right"); plt.legend(title="")
plt.tight_layout(); plt.savefig(FIG_DIR / "destvi_composition_by_condition.png", dpi=120); plt.show()

# 2. tumour burden -- positive control
fig, ax = plt.subplots(figsize=(5, 3))
ax.boxplot([obs.loc[cond == c, "ct_tumor_cell"].values for c in ORDER], labels=ORDER, showfliers=False)
ax.set(ylabel="tumor_cell proportion", title="Tumour proportion concentrates in CTCL")
plt.tight_layout(); plt.savefig(FIG_DIR / "destvi_tumor_burden.png", dpi=120); plt.show()

# 3. malignant signature by condition
fig, ax = plt.subplots(figsize=(5, 3))
ax.boxplot([obs.loc[cond == c, "malignant_score"].values for c in ORDER], labels=ORDER, showfliers=False)
ax.set(ylabel="malignant_overall score", title="Malignant signature higher in CTCL spots")
plt.tight_layout(); plt.savefig(FIG_DIR / "destvi_malignant_by_condition.png", dpi=120); plt.show()

# 4. per-section tumour burden (heterogeneity across sections)
sec = (obs.groupby("section", observed=True)
       .agg(tumor=("ct_tumor_cell", "mean"), condition=("condition", "first"))
       .sort_values("tumor", ascending=False))
colors = sec["condition"].map({"CTCL": "tab:red", "healthy": "tab:blue"})
ax = sec["tumor"].plot.bar(figsize=(7, 3), color=colors.values)
ax.set(ylabel="mean tumor_cell prop", xlabel="",
       title="Per-section tumour burden (red=CTCL, blue=healthy)")
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout(); plt.savefig(FIG_DIR / "destvi_tumor_by_section.png", dpi=120); plt.show()

# 5. subclonal-program usage in the tumour compartment, CTCL vs healthy
if prog_cols:
    prog = obs.groupby(cond, observed=True)[prog_cols].mean().T
    prog.index = [c.replace("program__", "") for c in prog_cols]
    ax = prog[ORDER].plot.bar(figsize=(7, 3.2))
    ax.set(ylabel="mean program score (tumour px_scale)", xlabel="",
           title="Subclonal-program usage in tumour compartment: CTCL vs healthy")
    plt.xticks(rotation=45, ha="right"); plt.legend(title="")
    plt.tight_layout(); plt.savefig(FIG_DIR / "destvi_programs_by_condition.png", dpi=120); plt.show()

## 5. Spatial annotation — all CTCL sections + healthy control

Full deconvolution panel per CTCL section: key cell-type proportions, dominant cell type, malignant
signature, and dominant subclonal program. One healthy section is rendered with the same panel as a
negative control (tumour + malignant signal should be near-absent). PNGs saved per section.

In [ ]:
# Full DestVI panel per CTCL section (+ one healthy negative control).
CT_KEYS = ["tumor_cell", "Th", "Treg", "CD8_T", "Myeloid", "Keratinocyte", "Fibroblast"]
PANEL = [f"ct_{g}" for g in CT_KEYS if f"ct_{g}" in vis.obs] + ["ct_argmax", "malignant_score", "program_argmax"]


def _title(c):
    if c.startswith("ct_") and c != "ct_argmax":
        return f"prop: {c[3:]}"
    return {"ct_argmax": "dominant cell type",
            "malignant_score": "malignant signature",
            "program_argmax": "dominant subclonal program"}.get(c, c)


def _annotate_section(s, size=1.5):
    sub = vis[vis.obs["section"] == s].copy()
    if sub.n_obs < 20:
        print(f"skip {s}: only {sub.n_obs} spots"); return
    sc.pl.spatial(sub, library_id=s, color=PANEL, ncols=4, size=size, wspace=0.25,
                  title=[_title(c) for c in PANEL], show=False)
    plt.suptitle(f"{s} ({sub.obs['condition'].iloc[0]}): DestVI deconvolution + subclonal programs",
                 y=1.005, fontsize=14)
    plt.savefig(FIG_DIR / f"destvi_annotated_{s}.png", dpi=140, bbox_inches="tight"); plt.show()


sc.settings.set_figure_params(figsize=(6, 6), dpi=90, facecolor="white")   # much larger panels
ctcl_sections = sorted(s for s in vis.obs["section"].cat.categories if str(s).startswith("CTCL"))
for s in ctcl_sections:
    _annotate_section(s)

healthy = [s for s in vis.obs["section"].cat.categories if str(s).startswith("WS_D_SKN")]
if healthy:                                   # one healthy section: tumour/malignant should be ~empty
    _annotate_section(healthy[0])
sc.settings.set_figure_params(figsize=(5, 3), dpi=80, facecolor="white")   # restore default

## Summary

- **CondSCVI → DestVI** (full shared gene space vs the Li-2024 scRNA atlas) → per-spot cell-type
  proportions (`obsm['ct_proportions']`, `obs['ct_<type>']`, `obs['ct_argmax']`) + nb-23 cross-patient
  subclonal **program** scores and `malignant_overall` on tumour-cell-specific expression
  (`obs['program_argmax']`, `obs['malignant_score']`) → `ctcl_visium_destvi.h5ad`.
- **CTCL vs healthy (§4):** tumour-cell proportion and malignant signature concentrate in the 8 CTCL
  sections vs the 15 healthy controls; cell-type composition and subclonal-program usage contrasted
  by condition.
- **Spatial (§5):** full deconvolution panel per CTCL section (+ one healthy control), saved to
  `figures/destvi_annotated_<section>.png`.
- **Caching:** CondSCVI/DestVI gated by `FORCE_TRAIN` (`condscvi_model/`, `destvi_model/`); derived
  h5ad gated by `FORCE_INFER`. A clean rerun loads everything and only redraws plots. Manual reload:
  `DestVI.load(DESTVI_DIR, vis)`.

**Next steps**
- Per-spot proportions unlock niche analysis: quantify tumour co-occurrence with Treg / Myeloid per
  CTCL section (spatial tumour–immune association).
- Contrast subclonal-program spatial layout (epidermotropism `skin_homing` vs egress
  `recirc_homing`) against histology.